# 内置Middleware - SummarizationMiddleware
- 作用：对历史会话概要总结,当做HumanMessage设置到历史对话最开始，减少上下文长度，控制成本
- 参数：
    - model: 概要模型
    - trigger: 触发条件，满足任意即可。类型：元组列表
        - tokens: 历史对话token数量，超过则触发
        - messages: 历史消息数量，超过则触发
        - fraction: 上下文长度比例。历史token累计达到 max_input_tokens * fraction 则触发。
            max_input_tokens通过profile设置，因此只要设置了改触发条件，如果模型model.profile是None，必须手动设置max_input_tokens，
            否则报错。这里的模型是指概要模型，而不是对话模型，因为Middleware只持有概要模型。所以当对话模型和概要模型不同时，要注意下fraction的
            设置，避免出现设置fraction为0.1，触发总结概要时，历史对话已经撑到了对话模型的50%，时机太晚。不过大部分场景，对话模型和概要模型设置为1个
    - keep: 保留的历史消息，支持三种策略（互斥）
        - tokens: 保留的历史对话tokens数量
        - messages: 保留的历史对话数量
        - fraction: 保留token数位max_input_tokens * fraction
    - summary_prompt: 自定义提示词，指示概要模型总结。默认使用内置提示词。自定义提示词必须有{messages} placeholder,使得历史消息列表可以被插入


In [11]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain.messages import HumanMessage, SystemMessage, AIMessage
from rich import print as rprint

from common import init_simple_dashscope_model

profile = {
    'max_input_tokens': 128_000
}
model = init_simple_dashscope_model('qwen-max')
summarization_model = init_simple_dashscope_model('deepseek-r1-distill-qwen-7b', profile=profile)

In [12]:

rprint(model.profile)
rprint(summarization_model.profile)


messages = [
    SystemMessage("你是个非常友好的AI助手"),
    HumanMessage("你好啊，我是老王，你是谁？"),
    AIMessage("你好老王，我是小王"),
    HumanMessage("好的小王，很高兴认识你"),
    AIMessage("你高兴得太早了"),
    HumanMessage("呵呵，你什么意思")
]

agent = create_agent(
    model=model,
    middleware=[
        SummarizationMiddleware(
            model=summarization_model,
            # 任意条件满足就出发
            trigger=[
                ('tokens', 10),
                ('messages', 12),
                ('fraction', 0.001)
            ],
            keep=('messages', 1),
            # keep=('tokens', 1),
            # keep=('tokens', 1),
            summary_prompt="对历史消息摘要，消息列表如下\n{messages}"
        ),
    ]
)

response = agent.invoke(
    {
        'messages': messages
    }
)

for msg in response['messages']:
    msg.pretty_print()


None

{'max_input_tokens': 128000}

================================ Human Message =================================

Here is a summary of the conversation to date:

你好老王，很高兴认识你！听你的故事真有趣，我猜你一定有很多地方值得分享。有什么我可以帮你的吗？
================================ Human Message =================================

呵呵，你什么意思
================================== Ai Message ==================================

看起来这里可能有一点小误会。您提到的“老王”可能是对我的一种称呼，但实际上我是Qwen，由阿里云开发的人工智能助手。如果我之前的信息让您产生了误解，还请您原谅。回到您的问题上，如果您有任何需要帮助的地方，无论是信息查询、学习资料寻找还是其他方面的问题，我都非常乐意为您提供帮助。请问具体有什么我可以协助您的吗？
